In [ ]:
# Now we have most optimized version of tf-idf with both for_idx and inv_idx and tf and idf
# But instead of using tf and idf (raw formula) we use BM formula given

# BM(t,d) =  SUMM idf(t,D) * [{tf(t,d)*(k1 + 1)}  / {tf(t,d) + k1 (1 - b + b*(doc_len / avg_doc_len))}]

# idf(t,D) =log10 [(N - df + 0.5)/ (df + 0.5)] <Negative output allowed> when df > N/2
# idf(t,D) =log10 [1 + (N - df + 0.5)/ (df + 0.5)] <Negative output NOT allowed>


# tf(t,d) = raw_counts of term t in doc d

In [15]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords , shakespeare
import math

In [16]:

nltk.download('shakespeare')
nltk.download('stopwords')

[nltk_data] Downloading package shakespeare to
[nltk_data]     C:\Users\Yatharth_Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package shakespeare is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yatharth_Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
def term_freq_unnorm_idx(corpus_name,lang):
    inverted_index = {}
    forward_index = {}
    file_lst = corpus_name.fileids()
    stopword = set(stopwords.words(lang))
    for id , doc in enumerate(file_lst):
        raw_words = corpus_name.words(fileid = doc)
        pro_words = [word.lower() for word in raw_words if word.lower() not in stopword]
        forward_index[id] = {}
        for word in pro_words:
            if word not in inverted_index : inverted_index[word] = {}
            inverted_index[word][id] = inverted_index[word].get(id,0) + 1
            
            forward_index[id][word] = forward_index[id].get(id,0) + 1
        
            
    return inverted_index ,forward_index, file_lst
        
inv_idx , for_idx , corpus = term_freq_unnorm_idx(shakespeare,'english')       


In [18]:
# The problem with this idf score with a floor function is this 
# The output can become -ve in nature and hence penalizes the docs with term q in query i.e if the doc is very short and majority of the
# words are term q then idf becomes -ve and other docs with no term q in them can score better than this doc even though it is unrelated
# when df > N/2 this problem arises 

In [19]:
def idf_(inv_idx,for_idx,word):
    if word not in inv_idx : return 0
    N = len(for_idx)
    doc_freq = len(inv_idx[word])
    idf_score = math.log10(1 + (N - doc_freq + 0.5)/(doc_freq + 0.5))
    
    return idf_score

In [47]:
def BMscore(inv_idx, for_idx, word, doc_id,avg_dl, k1=1.5, b=.75):
    if word not in inv_idx : return 0
    
    term_freq = for_idx[doc_id].get(word,0)
    
    if term_freq == 0 : return 0
    
    doc_len = sum(for_idx[doc_id].values())
    idf_score = idf_(inv_idx,for_idx,word)
    
    num = term_freq *(k1 + 1)
    dem = term_freq + k1 * (1 - b + b *(doc_len/avg_dl))
    
    return idf_score * (num/dem)
    

In [48]:
def BMsearch_one(query,inv_idx,for_idx,doc_id,avg_dl):
    score = 0
    for word in query:
        score += BMscore(inv_idx, for_idx, word, doc_id,avg_dl, k1=1.5, b=.75)
    return score
        


In [49]:
def BMsearch_all(query,inv_idx,for_idx,avg_dl):
    score_map = {}
    cand = set()
    
    for word in query:
        if word in inv_idx : cand |= set(inv_idx[word].keys())
        
    for doc_id in cand :
        score_map[doc_id] = BMsearch_one(query,inv_idx,for_idx,doc_id,avg_dl)
        
    return score_map
        
    

In [50]:

def rank_results(scores, corpus, k=10):
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return {corpus[doc_id]: score for doc_id, score in ranked[:k]}

In [51]:
def search(query, inv_idx,for_idx,corpus,avg_dl):
    pro_query = [w for w in query.lower().split() if w in inv_idx]
    scores = BMsearch_all(pro_query, inv_idx,for_idx,avg_dl)
    return rank_results(scores, corpus , k= 10)


In [52]:
# sum(for_idx[doc_id].values()) for one doc
avg_dl = 0
for doc_id in for_idx:
    avg_dl += sum(for_idx[doc_id].values())
avg_dl = avg_dl / len(for_idx)

In [55]:
a = search('Caesar othello brutus macbeth and dream of the romeo and juliet ',inv_idx,for_idx,corpus,avg_dl)

In [56]:
a

{'r_and_j.xml': 1.6100575106477701,
 'othello.xml': 1.043364691659504,
 'macbeth.xml': 1.0257928706631996,
 'j_caesar.xml': 0.6512777618707315,
 'a_and_c.xml': 0.5700559092114611,
 'hamlet.xml': 0.5179036539666986,
 'merchant.xml': 0.3956278184934653,
 'dream.xml': 0.08533061875834656}